### LLM Evaluation

This notebook compares the LLM output with the ground-truth dataset. It reports strict exact-match accuracy for `company_name` and `revenue`, macro set-based precision/recall/F1 for `countries` and `requested_coverages`, semantic inclusion accuracy for `industry`, and the malformed/failed record rate.

In [ ]:
from ast import literal_eval
from pathlib import Path

import pandas as pd

# Locate the CSVs from the repository root.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    DATA_DIR = candidate / "data" / "email_ingestion"
    if (DATA_DIR / "ground_truth.csv").exists():
        break
else:
    raise FileNotFoundError("Could not find data/email_ingestion/ground_truth.csv")

OUTPUT_DIR = DATA_DIR / "evaluation_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

ground_truth = pd.read_csv(DATA_DIR / "ground_truth.csv")
predictions = pd.read_csv(DATA_DIR / "llm_extracted_submissions.csv")

# Match one prediction row to each ground-truth row by submission ID.
comparison = ground_truth.merge(predictions, on="submission_id", how="left", suffixes=("_truth", "_pred"), indicator=True)

comparison["record_present"] = comparison["_merge"].eq("both")

# CSV parsing leaves list columns as strings. literal_eval handles both supplied
# representations: ["UK", "France"] and ['UK', 'France'].
def as_set(value):
    if pd.isna(value):
        return set()
    try:
        return set(literal_eval(value))
    except (ValueError, SyntaxError, TypeError):
        return set()


def list_scores(predicted, actual):
    predicted = as_set(predicted)
    actual = as_set(actual)
    true_positives = len(predicted & actual)
    precision = true_positives / len(predicted) if predicted else 0
    recall = true_positives / len(actual) if actual else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1

for field in ["countries", "requested_coverages"]:
    comparison[[f"{field}_precision", f"{field}_recall", f"{field}_f1"]] = comparison.apply(
        lambda row: list_scores(row[f"{field}_pred"], row[f"{field}_truth"]),
        axis=1,
        result_type="expand",
    )

# Strict scalar comparisons.
comparison["company_name_match"] = comparison["company_name_pred"] == comparison["company_name_truth"]
comparison["revenue_match"] = comparison["revenue_pred"] == comparison["revenue_truth"]

# Semantic industry match: accept either phrase containing the other.
comparison["industry_match"] = comparison.apply(
    lambda row: (
        isinstance(row["industry_pred"], str)
        and isinstance(row["industry_truth"], str)
        and (
            row["industry_pred"].casefold() in row["industry_truth"].casefold()
            or row["industry_truth"].casefold() in row["industry_pred"].casefold()
        )
    ),
    axis=1,
)

# A missing scalar or unparseable list is counted as a malformed/failed output.
comparison["malformed_or_failed"] = (
    ~comparison["record_present"]
    | comparison[["company_name_pred", "revenue_pred", "industry_pred"]].isna().any(axis=1)
    | comparison["countries_pred"].apply(lambda value: not isinstance(value, str) or not isinstance(literal_eval(value), list))
    | comparison["requested_coverages_pred"].apply(lambda value: not isinstance(value, str) or not isinstance(literal_eval(value), list))
)

metrics = pd.DataFrame([
    ["company_name", "Exact match accuracy (strict)", comparison["company_name_match"].mean()],
    ["revenue", "Exact match accuracy (strict)", comparison["revenue_match"].mean()],
    ["industry", "Semantic match accuracy", comparison["industry_match"].mean()],
    ["countries", "Macro precision", comparison["countries_precision"].mean()],
    ["countries", "Macro recall", comparison["countries_recall"].mean()],
    ["countries", "Macro F1", comparison["countries_f1"].mean()],
    ["requested_coverages", "Macro precision", comparison["requested_coverages_precision"].mean()],
    ["requested_coverages", "Macro recall", comparison["requested_coverages_recall"].mean()],
    ["requested_coverages", "Macro F1", comparison["requested_coverages_f1"].mean()],
    ["record", "Malformed / failed rate", comparison["malformed_or_failed"].mean()],
], columns=["field", "metric", "score"])
metrics["score_percent"] = (metrics["score"] * 100).round(2)

print(f"Ground-truth records: {len(ground_truth)}")
print(f"Prediction records: {len(predictions)}")
print(f"Malformed or failed records: {comparison['malformed_or_failed'].sum()}")
display(metrics[["field", "metric", "score_percent"]])

# Keep a concise error table for model debugging and interview discussion.
errors = comparison.loc[
    comparison["malformed_or_failed"]
    | ~comparison["company_name_match"]
    | ~comparison["revenue_match"]
    | ~comparison["industry_match"]
    | (comparison["countries_f1"] < 1)
    | (comparison["requested_coverages_f1"] < 1),
    [
        "submission_id", "malformed_or_failed",
        "company_name_truth", "company_name_pred",
        "revenue_truth", "revenue_pred",
        "countries_truth", "countries_pred",
        "industry_truth", "industry_pred",
        "requested_coverages_truth", "requested_coverages_pred",
    ],
]

metrics.to_csv(OUTPUT_DIR / "evaluation_metrics.csv", index=False)
errors.to_csv(OUTPUT_DIR / "evaluation_errors.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "evaluation_row_level.csv", index=False)

display(errors.head(20))

Ground-truth records: 50
Prediction records: 50
Malformed or failed records: 0


,field,metric,score_percent
0,company_name,Exact match accuracy (strict),98.00
1,revenue,Exact match accuracy (strict),94.00
2,industry,Semantic match accuracy,98.00
3,countries,Macro precision,98.67
4,countries,Macro recall,100.00
5,countries,Macro F1,99.00
6,requested_coverages,Macro precision,98.00
7,requested_coverages,Macro recall,98.00
8,requested_coverages,Macro F1,98.00
9,record,Malformed / failed rate,0.00


,submission_id,malformed_or_failed,company_name_truth,company_name_pred,revenue_truth,revenue_pred,countries_truth,countries_pred,industry_truth,industry_pred,requested_coverages_truth,requested_coverages_pred
6,E007,False,Foxmere Technologies Ltd,Foxmere Technologies Ltd,546984,488379,"[""Sweden"", ""Singapore"", ""United Kingdom""]","['Sweden', 'Singapore', 'United Kingdom']",Construction,Construction,"[""Media Liability""]",['Media Liability']
13,E014,False,Cedarwood UK Ltd,Cedarwood UK Ltd,28875880,28875880,"[""Denmark"", ""Germany""]","['Denmark', 'Germany']",Retail,Agriculture,"[""Professional Indemnity"", ""Cyber"", ""Technolog...","['Professional Indemnity', 'Cyber', 'Technolog..."
23,E024,False,Delta Labs Ltd,Delta Labs Ltd,42135291,42135291,"[""Ireland""]","['Ireland', 'France', 'Canada']",Life Sciences,Life Sciences,"[""Management Liability"", ""Property"", ""Media Li...","['Property', 'Management Liability', 'Media Li..."
38,E039,False,Glenhaven Holdings plc,Glenhaven Holdings plc,7237085,4848846,"[""Belgium"", ""United States""]","['Belgium', 'United States']",Education,Education,"[""Directors & Officers"", ""Management Liability...","['Management Liability', 'Technology E&O', 'Di..."
41,E042,False,Orchard Digital Ltd,Orchard Digital Ltd,8337689,8337689,"[""Netherlands""]",['Netherlands'],Real Estate,Real Estate,"[""General Liability""]",['Cyber']
46,E047,False,Maplecrest Industries Systems Ltd,Maplecrest Industries Ltd,33720755,33720755,"[""Australia"", ""Spain"", ""Denmark""]","['Australia', 'Spain', 'Denmark']",Construction,Construction,"[""Management Liability""]",['Management Liability']
49,E050,False,Upland Platforms Ltd,Upland Platforms Ltd,2008613,2510767,"[""Belgium"", ""Singapore"", ""United Kingdom""]","['Belgium', 'Singapore', 'United Kingdom']",Professional Services,Professional Services,"[""Management Liability"", ""Technology E&O""]","['Management Liability', 'Technology E&O']"
